In [13]:
import pyarrow.parquet as pq

import pandas as pd
import os
import json 
def read_data(path_prefix='data/'):
    # check this folder contains necessary parquet files
    if not (os.path.exists(f'{path_prefix}meta/movies.parquet')) or \
            not (os.path.exists(f'{path_prefix}meta/users_new.parquet')) or \
            not (os.path.exists(f'{path_prefix}ratings/ratings.parquet')) or \
            not (os.path.exists(f'{path_prefix}watches/watches.parquet')):
        raise FileNotFoundError("One or more required parquet files are missing in the specified path.")

    movies = pq.read_table(f'{path_prefix}meta/movies.parquet').to_pandas()
    users = pq.read_table(f'{path_prefix}meta/users_new.parquet').to_pandas()
    ratings = pq.read_table(f'{path_prefix}ratings/ratings.parquet').to_pandas()
    watches = pq.read_table(f'{path_prefix}watches/watches.parquet').to_pandas()

    selected_movie_cols = ['id', 'title', 'adult', 'budget', 'genres', 'original_language', 'overview', 'popularity', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'vote_average', 'vote_count']
    movies = movies[selected_movie_cols]
    movies['release_date'] = pd.to_datetime(movies['release_date'], errors='coerce')
    movies['release_year'] = movies['release_date'].dt.year
    movies.rename(columns={'id': 'movie_id'}, inplace=True)
    

    def parse_name_field(value):
        if value is None or (isinstance(value, float) and pd.isna(value)):
            return []
        parsed = value
        if isinstance(value, str):
            value = value.strip()
            if not value:
                return []
            try:
                parsed = json.loads(value)
            except json.JSONDecodeError:
                return [value]
        if isinstance(parsed, list):
            names = []
            for item in parsed:
                if isinstance(item, dict) and "name" in item:
                    names.append(item["name"])
                elif isinstance(item, str):
                    names.append(item)
            return names
        if isinstance(parsed, dict) and "name" in parsed:
            return [parsed["name"]]
        return []
    movies['genres'] = movies['genres'].apply(parse_name_field)
    movies['production_companies'] = movies['production_companies'].apply(parse_name_field)
    movies['production_countries'] = movies['production_countries'].apply(parse_name_field)

    numeric_features = ['budget', 'popularity', 'revenue', 'runtime', 
                           'vote_average', 'vote_count']
    for feature in numeric_features:
        movies[feature] = pd.to_numeric(movies[feature], errors='coerce')

    all_features = ['adult', 'budget', 'genres', 'original_language', 'overview',
                    'popularity', 'production_companies', 'production_countries', 
                    'release_date', 'release_year'] + numeric_features 
    assert all(feature in movies.columns for feature in all_features)
    # convert adult to boolean
    movies['adult'] = movies['adult'].astype(bool)
    return movies, users, ratings, watches

movies, users, ratings, watches = read_data('../../data-pull/data/')

print(f"movies: {movies.columns.tolist()}")
print(f"users: {users.columns.tolist()}")
print(f"ratings: {ratings.columns.tolist()}")
print(f"watches: {watches.columns.tolist()}")
print(f'Movies shape: {movies.shape}    Users shape: {users.shape}    Ratings shape: {ratings.shape}    Watches shape: {watches.shape}')


movies: ['movie_id', 'title', 'adult', 'budget', 'genres', 'original_language', 'overview', 'popularity', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'vote_average', 'vote_count', 'release_year']
users: ['user_id', 'age', 'occupation', 'gender']
ratings: ['timestamp', 'user_id', 'movie_id', 'rating']
watches: ['user_id', 'movie_id', 'timestamp_start', 'timestamp_end', 'minutes_watched']
Movies shape: (26718, 16)    Users shape: (1000000, 4)    Ratings shape: (543534, 4)    Watches shape: (879171, 5)


In [16]:
print(min(watches['timestamp_start'].min() , ratings['timestamp'].min()))
max(watches['timestamp_end'].max() if not watches.empty else pd.Timestamp.min(), ratings['timestamp'].max() if not ratings.empty else pd.Timestamp.min())


2025-11-12 00:01:56+00:00


Timestamp('2025-11-13 02:59:34.131950+0000', tz='UTC')

In [20]:
# join users, ratings and movies
ratings_meta = ratings.merge(users, on='user_id').merge(movies, on='movie_id')
watches_meta = watches.merge(users, on='user_id').merge(movies, on='movie_id')

In [8]:
ratings_meta.head()

,timestamp,user_id,movie_id,rating,age,occupation,gender,title,adult,budget,...,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,vote_average,vote_count,release_year
0,2025-08-27 14:50:35,142245,a+mighty+wind+2003,3,33,homemaker,F,A Mighty Wind,False,6000000,...,"In ""A Mighty Wind"", director Christopher Guest...",4.672036,[Castle Rock Entertainment],[United States of America],2003-04-16,18750246,91,6.6,75,2003
1,2025-08-27 15:54:56,90741,carlitos+way+1993,4,26,college/grad student,M,Carlito's Way,False,30000000,...,"A Puerto-Rican ex-con, just released from pris...",8.698509,"[Universal Pictures, Epic Productions, Bregman...",[United States of America],1993-11-10,36516012,144,7.7,805,1993
2,2025-08-27 16:28:42,8551,beverly+hills+cop+iii+1994,3,29,executive/managerial,M,Beverly Hills Cop III,False,50000000,...,Back in sunny southern California and on the t...,11.787784,"[Paramount Pictures, Eddie Murphy Productions]",[United States of America],1994-05-24,119208989,104,5.5,445,1994
3,2025-08-27 17:12:06,24772,jesus+camp+2006,3,9,artist,M,Jesus Camp,False,0,...,A growing number of Evangelical Christians bel...,4.886956,[],[United States of America],2006-09-15,0,87,6.6,119,2006
4,2025-08-27 17:39:15,2432,nell+1994,5,27,executive/managerial,M,Nell,False,31000000,...,"In a remote woodland cabin, a small town docto...",5.930957,"[Twentieth Century Fox Film Corporation, Egg P...",[United States of America],1994-12-23,106683817,112,6.1,128,1994


In [9]:
watches_meta.head()

,user_id,movie_id,timestamp_start,timestamp_end,minutes_watched,age,occupation,gender,title,adult,...,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,vote_average,vote_count,release_year
0,770,the+godfather+part+ii+1974,2025-09-22 04:54:28,2025-09-22 04:54:28,1,34,sales/marketing,F,The Godfather: Part II,False,...,In the continuing saga of the Corleone crime f...,36.629307,"[Paramount Pictures, The Coppola Company]",[United States of America],1974-12-20,47542841,200,8.3,3418,1974
1,954,sal_+or+the+120+days+of+sodom+1975,2025-09-22 04:54:28,2025-09-22 04:54:28,1,33,executive/managerial,M,"Salò, or the 120 Days of Sodom",False,...,Four corrupted fascist libertines round up 9 t...,7.705479,"[United Artists, Les Productions Artistes Asso...","[Italy, France]",1975-11-22,0,116,6.4,332,1975
2,1576,airplane+1980,2025-08-06 08:46:51,2025-08-06 10:14:46,49,23,college/grad student,M,Airplane!,False,...,"Alcoholic pilot, Ted Striker has developed a f...",13.063203,[Paramount Pictures],[United States of America],1980-07-02,83453539,88,7.1,1104,1980
3,1576,shrek+2001,2025-08-14 00:40:44,2025-08-14 00:46:53,4,23,college/grad student,M,Shrek,False,...,It ain't easy bein' green -- especially if you...,17.987728,"[DreamWorks SKG, Pacific Data Images (PDI), Dr...",[United States of America],2001-05-16,484409218,90,7.3,4183,2001
4,1576,star+wars+episode+i+-+the+phantom+menace+1999,2025-07-28 23:49:56,2025-07-29 02:08:05,76,23,college/grad student,M,Star Wars: Episode I - The Phantom Menace,False,...,"Anakin Skywalker, a young slave strong with th...",15.649091,[Lucasfilm],[United States of America],1999-05-19,924317558,136,6.4,4526,1999


In [4]:
import altair as alt
# visualize user age by occupation and gender
alt.Chart(ratings_meta).mark_bar().encode(
    x='count()',
    y='age',
    color='gender',
    tooltip=['age', 'gender', 'count()']
).properties(
    title='User Age Distribution by Gender'
).interactive()

alt.Chart(...)

In [5]:
# visualize user occupation distribution
alt.Chart(ratings_meta).mark_bar().encode(
    x='count()',
    y='occupation', 
    tooltip=['occupation', 'count()']
).properties(
    title='User Occupation Distribution'
).interactive()

alt.Chart(...)

In [7]:
# visualize watch distribution breakdown by gender and movie genre
alt.Chart(watches_meta.explode('genres')).mark_bar().encode(
    x='count()',
    y='genres', 
    color='gender',
    tooltip=['genres', 'gender', 'count()']
).properties(
    title='Watch Distribution by Gender and Genre'
).interactive()

alt.Chart(...)